In [1]:
# Importer och diverse
import os
import json
from google import genai
from dotenv import load_dotenv
from google.genai import types
import numpy as np
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

## Chunking
Filer över 1800 tecken chunkades, medan övriga lessons behölls som naturliga semantiska enheter. Detta val gjordes för att balansera kontextbevarande mot precision i retrieval.

In [ ]:
def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = end - overlap

    return chunks


def load_all_courses_with_chunking(base_folder, chunk_threshold=1800, chunk_size=1200, overlap=200):
    documents = []

    # Gå igenom alla undermappar, t.ex. cleaned/3064200, cleaned/1234567 osv.
    for course_folder in os.listdir(base_folder):
        course_path = os.path.join(base_folder, course_folder)

        if not os.path.isdir(course_path):
            continue

        for file in os.listdir(course_path):
            if not file.endswith(".json"):
                continue

            file_path = os.path.join(course_path, file)

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            text = data.get("text", "").strip()

            # Filtrera bort nästan tomma / väldigt tunna texter
            if len(text) < 20 or len(text.split()) < 5:
                continue

            # Om texten är liten nog: behåll som en chunk
            if len(text) <= chunk_threshold:
                split_texts = [text]
            else:
                split_texts = chunk_text(
                    text,
                    chunk_size=chunk_size,
                    overlap=overlap
                )

            # Lägg till varje chunk som ett eget dokument
            for i, chunk in enumerate(split_texts):
                documents.append({
                    "text": chunk,
                    "metadata": {
                        "course_id": data.get("course_id"),
                        "course_name": data.get("course_name"),
                        "chapter_id": data.get("chapter_id"),
                        "chapter_title": data.get("chapter_title"),
                        "lesson_id": data.get("lesson_id"),
                        "lesson_title": data.get("lesson_title"),
                        "file": file,
                        "chunk_index": i,
                        "chunk_count": len(split_texts),
                        "original_length": len(text)
                    }
                })

    return documents

In [3]:
base_folder = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned"

documents = load_all_courses_with_chunking(
    base_folder=base_folder,
    chunk_threshold=1800,
    chunk_size=1200,
    overlap=200
)

print(f"Antal dokument/chunks: {len(documents)}")
print("\nExempel:")
print(documents[0]["metadata"])
print(documents[0]["text"])

Antal dokument/chunks: 437

Exempel:
{'course_id': '1703530', 'course_name': 'Ansvariga- Vem ska jag kontakta?', 'chapter_id': '7643418', 'chapter_title': 'Vem ska jag kontakta?', 'lesson_id': '31459054', 'lesson_title': 'Båtskada', 'file': '31459054.json', 'chunk_index': 0, 'chunk_count': 1, 'original_length': 749}
Olika skeppare har olika erfarenhet av olika båtar, var därför inte rädd för att fråga föregående skeppare om saker du undrar över. Frågor om båten bör du ställa redan på bytesdagen och självklart kan du alltid höra av dig under veckan om du har ytterligare frågor. Vet du inte vem som varit på båten tidigare eller har föregående skepparen inte svar på dina frågor? Då skriver du i skeppargruppen eller en av dina mentorer.
Skeppargruppen skall kunna användas för att fråga om eventuella strul, väder eller generella frågor. Om du mot förmodan inte får ett vettigt svar är det bäst att kontakta din
platsansvarig.
Har du en skada efter till exempel en tilläggning eller liknande sk

Här kollar jag hur många dokument som blev chunkade

In [4]:
chunked_docs = [doc for doc in documents if doc["metadata"]["chunk_count"] > 1]

print(f"Antal chunkade dokument: {len(chunked_docs)}")

for doc in chunked_docs[:10]:
    print(
        doc["metadata"]["file"],
        "| chunk",
        doc["metadata"]["chunk_index"] + 1,
        "av",
        doc["metadata"]["chunk_count"],
        "| originallängd:",
        doc["metadata"]["original_length"]
    )

Antal chunkade dokument: 188
60039047.json | chunk 1 av 4 | originallängd: 3214
60039047.json | chunk 2 av 4 | originallängd: 3214
60039047.json | chunk 3 av 4 | originallängd: 3214
60039047.json | chunk 4 av 4 | originallängd: 3214
60039176.json | chunk 1 av 3 | originallängd: 2502
60039176.json | chunk 2 av 3 | originallängd: 2502
60039176.json | chunk 3 av 3 | originallängd: 2502
64247725.json | chunk 1 av 3 | originallängd: 2929
64247725.json | chunk 2 av 3 | originallängd: 2929
64247725.json | chunk 3 av 3 | originallängd: 2929


In [5]:
chunked_files = set()

for doc in documents:
    if doc["metadata"]["chunk_count"] > 1:
        key = (
            doc["metadata"]["course_id"],
            doc["metadata"]["file"]
        )
        chunked_files.add(key)

print("Antal unika filer som chunkades:", len(chunked_files))

Antal unika filer som chunkades: 60


## Embeddings
Här testade jag först med gemini embedding men blev snabbt problem med request limits. Detta löste jag igenom att sätta endast köra 95 chunks i taget och sedan vänta 65 sekunder. Då kom jag under gratis versionens gränser och slapp köra det på en lokal modell.

In [7]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
def create_embeddings(text, model="gemini-embedding-001", task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

def embed_documents(documents):
    embedded_docs = []

    for doc in documents:
        response = create_embeddings(doc["text"])
        embedding = response.embeddings[0].values

        embedded_docs.append({
            "text": doc["text"],
            "embedding": embedding,
            "metadata": doc["metadata"]
        })

    return embedded_docs

In [9]:
for i, doc in enumerate(documents):
    print(f"\n--- Dokument {i+1} ---")
    print("Fil:", doc["metadata"]["file"])
    print("Längd:", len(doc["text"]))


--- Dokument 1 ---
Fil: 31459054.json
Längd: 749

--- Dokument 2 ---
Fil: 31459057.json
Längd: 442

--- Dokument 3 ---
Fil: 31459076.json
Längd: 931

--- Dokument 4 ---
Fil: 31459077.json
Längd: 585

--- Dokument 5 ---
Fil: 31460254.json
Längd: 686

--- Dokument 6 ---
Fil: 31460310.json
Längd: 566

--- Dokument 7 ---
Fil: 31460350.json
Längd: 283

--- Dokument 8 ---
Fil: 31460404.json
Längd: 257

--- Dokument 9 ---
Fil: 60038464.json
Längd: 786

--- Dokument 10 ---
Fil: 60039047.json
Längd: 1200

--- Dokument 11 ---
Fil: 60039047.json
Längd: 1200

--- Dokument 12 ---
Fil: 60039047.json
Längd: 1200

--- Dokument 13 ---
Fil: 60039047.json
Längd: 213

--- Dokument 14 ---
Fil: 60039176.json
Längd: 1199

--- Dokument 15 ---
Fil: 60039176.json
Längd: 1200

--- Dokument 16 ---
Fil: 60039176.json
Längd: 502

--- Dokument 17 ---
Fil: 64247538.json
Längd: 1658

--- Dokument 18 ---
Fil: 64247725.json
Längd: 1200

--- Dokument 19 ---
Fil: 64247725.json
Längd: 1200

--- Dokument 20 ---
Fil: 642477

In [10]:
def create_embeddings(text, model="gemini-embedding-001", task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

def embed_documents(documents):
    embedded_docs = []

    for doc in documents:
        response = create_embeddings(doc["text"])
        embedding = response.embeddings[0].values

        embedded_docs.append({
            "text": doc["text"],
            "embedding": embedding,
            "metadata": doc["metadata"]
        })

    return embedded_docs

In [12]:
import time

def embed_documents_with_delay(documents, batch_size=95, wait_seconds=65):
    embedded_docs = []

    for i, doc in enumerate(documents, start=1):
        print(f"Embedding {i}/{len(documents)}")

        response = create_embeddings(doc["text"])
        embedding = response.embeddings[0].values

        embedded_docs.append({
            "text": doc["text"],
            "embedding": embedding,
            "metadata": doc["metadata"]
        })

        # Vänta efter varje batch
        if i % batch_size == 0 and i < len(documents):
            print(f"\nBatch klar. Väntar {wait_seconds} sekunder...\n")
            time.sleep(wait_seconds)

    return embedded_docs

In [13]:
embedded_docs = embed_documents_with_delay(documents, batch_size=95, wait_seconds=65)

print(f"Antal embeddings: {len(embedded_docs)}")
print(f"Dimension på första embedding: {len(embedded_docs[0]['embedding'])}")

Embedding 1/437
Embedding 2/437
Embedding 3/437
Embedding 4/437
Embedding 5/437
Embedding 6/437
Embedding 7/437
Embedding 8/437
Embedding 9/437
Embedding 10/437
Embedding 11/437
Embedding 12/437
Embedding 13/437
Embedding 14/437
Embedding 15/437
Embedding 16/437
Embedding 17/437
Embedding 18/437
Embedding 19/437
Embedding 20/437
Embedding 21/437
Embedding 22/437
Embedding 23/437
Embedding 24/437
Embedding 25/437
Embedding 26/437
Embedding 27/437
Embedding 28/437
Embedding 29/437
Embedding 30/437
Embedding 31/437
Embedding 32/437
Embedding 33/437
Embedding 34/437
Embedding 35/437
Embedding 36/437
Embedding 37/437
Embedding 38/437
Embedding 39/437
Embedding 40/437
Embedding 41/437
Embedding 42/437
Embedding 43/437
Embedding 44/437
Embedding 45/437
Embedding 46/437
Embedding 47/437
Embedding 48/437
Embedding 49/437
Embedding 50/437
Embedding 51/437
Embedding 52/437
Embedding 53/437
Embedding 54/437
Embedding 55/437
Embedding 56/437
Embedding 57/437
Embedding 58/437
Embedding 59/437
Embedd

In [4]:
# Funktion som räknar ut cosinuslikheten

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [5]:
# En sökfunktion

def search_documents(query, embedded_docs, top_k=3):
    query_response = create_embeddings(query)
    query_embedding = query_response.embeddings[0].values

    scored_docs = []

    for doc in embedded_docs:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored_docs.append({
            "score": score,
            "text": doc["text"],
            "metadata": doc["metadata"]
        })

    scored_docs = sorted(scored_docs, key=lambda x: x["score"], reverse=True)
    return scored_docs[:top_k]

Testar retrival utan LLM

In [16]:
results = search_documents("Vad betyder att reva?", embedded_docs, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Träff {i} ---")
    print("Score:", result["score"])
    print("Lesson:", result["metadata"]["lesson_title"])
    print("Fil:", result["metadata"]["file"])
    print("Text:")
    print(result["text"])


--- Träff 1 ---
Score: 0.8146748726077576
Lesson: Terminologi segling
Fil: 11511019.json
Text:
Storsegel = Det aktre, större, seglet
Genua = Förseglet som är rullat på förstaget
Sätta segel = Hissa upp seglen
Reva segel = Minska segelytan
Vindöga = När båten ligger så att vinden kommer rakt framifrån
Fall = Används för att hissa upp seglet
Skot = Används för att reglera seglets form
Skota = Tar hem på seglet för att ändra formen på det
Lazyjacks = Tampar som går från masten till lazybagen för att samla ihop seglet i lazybagen
Lazybag = Säcken som sitter på bommen som skyddar seglet när det inte är hissat
Avlastare = Låsning för tampar (kallas i vardagstal ibland för spinlock)
Furlex = System för att rulla in förseglet runt förstaget, trumman sitter antingen under däck eller ovan däck
Furlexlina = Tampen som sitter på furlexen, den släpps för att rulla ut och dras för att rulla in
Vinsch = Ett system på båten som används för att justera segel när det är tungt
Vinscha = Man vevar med et

## Bygger modellen
Bygger prompten för modellen

In [2]:
def build_prompt(query, results):
    context = "\n\n".join(
        [f"Lektion: {r['metadata']['lesson_title']}\n{r['text']}" for r in results]
    )

    prompt = f"""
Du är en hjälpsam assistent för More Sailings e-learning.

Använd endast informationen i kontexten nedan.
Om svaret inte finns i kontexten, säg att du inte vet baserat på materialet.

KONTEXT:
{context}

FRÅGA:
{query}
"""
    return prompt

In [21]:
query = "Vad betyder att reva och hur ska man tänka när man revar?"
results = search_documents(query, embedded_docs, top_k=3)
prompt = build_prompt(query, results)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Att reva innebär att man minskar segelytan.

När man revar ska man tänka på följande:
*   Det måste alltid ske växelvis för att man ska få balans i båten och hålla ett så jämnt tryck i masten som möjligt.
*   Man kan till exempel ta tre rev i genuan samtidigt som man har två rev i storen. Detta gör det snabbare att öka segelytan om vinden minskar.
*   Genuan ger tryck framåt och storen ger tryck bakåt i masten.
*   När storen revas flyttas tryckpunkten nedåt längs masten, vilket gör att man måste spänna akterstaget för att masttoppen inte ska dras framåt och knäckas.
*   Man kan inte segla med flera rev i storen och inget rev i genuan, då blir balansen fel.
*   Om man ska ta flera rev i storseglet, måste man ta det första revet innan man kan ta det andra revet, och man måste även ta 1-3 rev i genuan.


In [22]:
query = "Jag har problem med min båt vem ska jag ringa?"
results = search_documents(query, embedded_docs, top_k=3)
prompt = build_prompt(query, results)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Vem du ska kontakta beror på typen av problem:

*   **För allmänna frågor eller om du undrar över något:** Fråga föregående skeppare. Om de inte kan svara, vänd dig till skeppargruppen eller en av dina mentorer. Får du inget vettigt svar från skeppargruppen, kontakta din platsansvarig.
*   **För mindre skador eller problem som du själv inte kan åtgärda (efter egen felsökning):** Vänd dig till dina kollegor och dina mentorer för hjälp. Alla saker som går sönder ska rapporteras till en mentor. Du ska även skriva upp dem och skicka till en ansvarig som meddelar charterbolaget.
*   **Vid en skada efter till exempel en tilläggning:** Kontakta din platsansvarig samt ansvarig tekniker för din båt.
*   **Vid större skador eller problem med båten, eller vid olyckor/grundstötning (som ska informeras om direkt):** Kontakta servicepersonal hos charterbolaget omedelbart.
*   **Vid skador som inte är akuta men kräver reservdelar:** Kontakta personalen hos charterbolaget för att förbereda dem och ge 

In [23]:
query = "Vad finns det att göra på Hvar och i Hvar stad?"
results = search_documents(query, embedded_docs, top_k=3)
prompt = build_prompt(query, results)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Baserat på materialet finns det följande att göra på Hvar och i Hvar stad:

*   **Mat och dryck:**
    *   Hitta ett stort utbud av restauranger, caféer och barer.
    *   Besök Hula Hula beach bar, en strandklubb med bra mat och en upplevelse.
    *   Ät på rekommenderade restauranger som Kapetana (kött och fisk), Fig cafe bar (mexikanska maträtter, vegetariska alternativ), Lola - street food och Caperasa.
    *   Drick på ställen som Drink Ka'lavanda.
*   **Kultur och shopping:**
    *   Besök ateljéer i de trånga gränderna där lokala konstnärer ställer ut sina verk.
    *   Handla i supermarket.
    *   Besök en bagare.
*   **Sevärdheter och upplevelser:**
    *   Se Fortciafäsningen, även kallat Spanska fortet, som vakar ner över staden.
    *   Upplev Hvar som är en av Kroatiens största turistmål.
    *   Se Hvar som är känt för sina odlingar av lavendel.
    *   Besök turistbyrån som är en av Europas äldsta med start 1868.
    *   Njut av att vara på en av Europas solsäkraste pla

In [24]:
query = "Hur ser rutinerna ut kring mathantering?"
results = search_documents(query, embedded_docs, top_k=3)
prompt = build_prompt(query, results)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Rutinerna kring mathantering, baserat på materialet, ser ut som följer:

**Personlig hygien vid mathantering:**
*   Tvätta händerna i minst 20 sek och använd alcogel ofta.
*   Använd förkläde.
*   Långt hår ska vara noga uppsatt.
*   Inte ha långa, målade naglar eller smycken.
*   Det är en fördel att använda handskar vid hantering av kött och kyckling.

**Allmän livsmedelshygien:**
*   Kontrollera att bäst före datum inte har passerat och att råvarorna ser bra ut och luktar fräscht.
*   Förvara maten rätt ombord för att kvalitén ska hålla väl.
*   Se till att kylskåpen och eventuell frys är tillräckligt kalla (Kyl +4 grader, Frys -18 grader). På båt innebär detta ofta att kyl/frys ska vara inställt på kallast möjliga.
*   Diska alla redskap som skärbrädor, knivar, serveringsfat ordentligt.
*   Se till att köket är rent, hygieniskt och fräscht.
*   Var noga med att diska alla redskap, tallrikar, glas och bestick ordentligt.
*   Torka av alla ytor, skåp och kylskåp regelbundet.
*   Anvä